# Toxic financing: notes that convert below the lowest trading price

Search filings for conversion prices set at a discount to the lowest recent price, the terms behind death-spiral dilution.

Read it with charts at https://www.quantisnow.com/sec-api/guides/toxic-convertible-financing. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

"Convertible at 75% of the lowest trading price in the prior 10 trading days" means the lender profits more the further the stock falls, and every conversion adds shares that push it lower. These terms show up in small-cap 10-Qs, 8-Ks and resale S-1s.

## Conversion prices tied to the lowest trading price

### Which filings describe notes that convert at a discount to the lowest trading price?

A conversion price tied to the lowest recent trading price rewards the lender for pushing the stock down. It's the clearest sign of a death-spiral financing.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"conversion price\" AND (\"lowest trading price\" OR \"lowest closing bid price\" OR \"lowest daily VWAP\" OR \"lowest VWAP\")",
  "formTypes": ["10-Q", "10-K", "8-K", "S-1", "S-1/A"],
  "sort": "filedAt:desc",
  "size": 5
}'
```

Response (real, first 2 of 54):

```json
{
  "total": {
    "value": 54,
    "relation": "eq"
  },
  "took": 116,
  "data": [
    {
      "id": "0001477932-26-007251",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001477932-26-007251",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:21-05:00",
      "cik": "0001425627",
      "companyName": "SOBR Safe Inc.",
      "ticker": "SOBR",
      "tickers": [
        "SOBR"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SOBR Safe Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1425627/000147793226007251/0001477932-26-007251-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-sobr-safe-inc-6804398",
      "score": null,
      "highlights": [
        "Pursuant to the Inducement Letters, the Holders agreed to convert some or all of the Applicable Notes at a reduced <em>conversion price</em> equal to $682 per share (such reduced <em>conversion price</em>, the \"Notes <em>Conversion Price</em>\").",
        "In connection with such conversion, the Notes <em>Conversion Price</em> was permanently reduced to $682 . The Company recognized conversion expense of $ 585,875 for the induced conversion.",
        "The <em>lowest VWAP</em> during the trading period of $ 4.71 was determined to be below the Floor Price resulting in a True-up Payment due to the remaining holders of $ 1,640,000 ."
      ]
    },
    {
      "id": "0001096906-26-001885",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001096906-26-001885",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:23:25-05:00",
      "cik": "0001854275",
      "companyName": "Zoomcar Holdings Inc.",
      "ticker": "ZCAR",
      "tickers": [
        "ZCAR"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by Zoomcar Holdings Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1854275/000109690626001885/0001096906-26-001885-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-zoomcar-holdings-inc-6804405",
      "score": null,
      "highlights": [
        "The note becomes convertible into common shares after 180 days or upon an event of default, at a <em>conversion price</em> equal to 73 % of the <em>lowest closing bid price</em> over the preceding 15 trading days, subject to a 4.99 % (up to",
        "<em>price</em> equal to 73 - 75 % of the <em>lowest trading price</em> of the Company's Common stock during the ten or fifteen trading days (as specified in the Note agreement) immediately prior to the applicable conversion date.",
        "\"Market Price\" shall mean 75 % of the <em>lowest closing bid price</em> of the Common Stock on the Principal Market during the fifteen ( 15 ) Trading Day period immediately preceding the respective Conversion Date."
      ]
    }
  ]
}
```

## In Python

In [ ]:
import os, requests

res = requests.post(
    "https://www.quantisnow.com/api/v1/full-text",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={
        "query": "\"conversion price\" AND (\"lowest trading price\" OR \"lowest closing bid price\" OR \"lowest daily VWAP\" OR \"lowest VWAP\")",
        "formTypes": ["10-Q","10-K","8-K","S-1","S-1/A"],
        "sort": "filedAt:desc",
        "size": 50,
    },
    timeout=30,
)
res.raise_for_status()
for f in res.json()["data"]:
    print(f["filedAt"][:10], f["ticker"], f["formType"], f["companyName"])
    if f["highlights"]:
        print("   ", f["highlights"][0].replace("<em>", "").replace("</em>", ""))

## From the command line

```shell
npx quantisnow search '"conversion price" AND ("lowest trading price" OR "lowest VWAP")' --newest
```

Look for the discount (70% to 85% is typical), the lookback window, and whether conversion starts right away or only after a default. An S-1 registering the conversion shares for resale usually follows.

Related: [Convertible notes](https://www.quantisnow.com/sec-api/guides/convertible-notes-8-k), [Reverse splits](https://www.quantisnow.com/sec-api/guides/reverse-stock-splits), [Going-concern warnings](https://www.quantisnow.com/sec-api/guides/find-going-concern-companies).